# Customer Churn — Final Model Evaluation and Persistence

## Objective

Evaluate the tuned customer churn models on the untouched test set and prepare the selected model for deployment.

## Models

- Logistic Regression
- Random Forest

## Evaluation Metrics

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC
- Confusion Matrix

## Important

The test set was not used during feature investigation or hyperparameter tuning.

It is used only for final model evaluation.

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

import os

In [2]:
data = pd.read_csv(
    "../data/processed/telco_customer_churn_cleaned.csv"
)

print("Dataset shape:", data.shape)

Dataset shape: (7043, 26)


In [3]:
target_column = "Churn Label"

selected_features = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "CLTV",
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method"
]

X = data[selected_features]
y = data[target_column]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (7043, 20)
Target shape: (7043,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True) * 100)

X_train: (5634, 20)
X_test: (1409, 20)

Training target distribution:
Churn Label
No     73.464679
Yes    26.535321
Name: proportion, dtype: float64

Test target distribution:
Churn Label
No     73.456352
Yes    26.543648
Name: proportion, dtype: float64


In [5]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']

Categorical features: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


C:\Users\Tanish_Gupta\AppData\Local\Temp\ipykernel_5192\144249423.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [6]:
def create_preprocessor():

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features)
        ]
    )

    return preprocessor

In [7]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor()),
        ("model", LogisticRegression(
            C=10,
            class_weight=None,
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [8]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=10,
            min_samples_split=2,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [9]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model
}

trained_models = {}

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

    trained_models[name] = model

    print(f"{name} completed.\n")

Training Logistic Regression...
Logistic Regression completed.

Training Random Forest...
Random Forest completed.



In [10]:
results = []

predictions = {}
probabilities = {}

for name, model in trained_models.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    predictions[name] = y_pred
    probabilities[name] = y_prob

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test,
            y_pred,
            pos_label="Yes"
        ),
        "Recall": recall_score(
            y_test,
            y_pred,
            pos_label="Yes"
        ),
        "F1": f1_score(
            y_test,
            y_pred,
            pos_label="Yes"
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            y_prob
        ),
        "PR-AUC": average_precision_score(
            (y_test == "Yes").astype(int),
            y_prob
        )
    })

final_results = pd.DataFrame(results)

final_results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.797729,0.632047,0.569519,0.599156,0.848097,0.639576
1,Random Forest,0.803407,0.654952,0.548128,0.596798,0.851578,0.668773


In [11]:
for name in trained_models:

    cm = confusion_matrix(
        y_test,
        predictions[name]
    )

    print("=" * 50)
    print(name)
    print("=" * 50)

    print(cm)

Logistic Regression
[[911 124]
 [161 213]]
Random Forest
[[927 108]
 [169 205]]


In [12]:
baseline_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Baseline ROC-AUC": [
        0.833698,
        0.842000
    ]
})

tuned_comparison = final_results.merge(
    baseline_results,
    on="Model"
)

tuned_comparison[
    [
        "Model",
        "Baseline ROC-AUC",
        "ROC-AUC"
    ]
]

,Model,Baseline ROC-AUC,ROC-AUC
0,Logistic Regression,0.833698,0.848097
1,Random Forest,0.842000,0.851578


In [13]:
final_results.sort_values(
    by="ROC-AUC",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
1,Random Forest,0.803407,0.654952,0.548128,0.596798,0.851578,0.668773
0,Logistic Regression,0.797729,0.632047,0.569519,0.599156,0.848097,0.639576


In [ ]:
final_model_name = "Random Forest"
final_model = trained_models[final_model_name]
print("Final model:", final_model_name)

Final model: Random Forest


In [15]:
import os

os.makedirs("../models", exist_ok=True)

print("Models directory is ready.")

Models directory is ready.


In [16]:
model_path = "../models/customer_churn_model.joblib"

joblib.dump(
    final_model,
    model_path
)

print("Final model saved successfully.")
print("Path:", model_path)

Final model saved successfully.
Path: ../models/customer_churn_model.joblib


In [17]:
loaded_model = joblib.load(model_path)

sample_data = X_test.iloc[:5]

sample_predictions = loaded_model.predict(sample_data)
sample_probabilities = loaded_model.predict_proba(sample_data)[:, 1]

print("Sample predictions:")
print(sample_predictions)

print("\nSample churn probabilities:")
print(sample_probabilities)

Sample predictions:
['No' 'Yes' 'No' 'No' 'No']

Sample churn probabilities:
[0.03665955 0.70424077 0.08404888 0.4511946  0.03745402]


In [18]:
print("Loaded model type:", type(loaded_model).__name__)
print("Pipeline steps:", loaded_model.named_steps.keys())

Loaded model type: Pipeline
Pipeline steps: dict_keys(['preprocessor', 'model'])


## Step 11 Conclusion

The tuned customer churn models were evaluated on the untouched test set.

Random Forest achieved the highest ROC-AUC and PR-AUC among the evaluated models, while Logistic Regression achieved higher churn recall and F1-score.

Using ROC-AUC as the primary model-selection criterion, Random Forest was selected as the final Customer Churn model.

The complete preprocessing and model pipeline was persisted as:

`models/customer_churn_model.joblib`

The persisted pipeline is ready for integration into the Customer Intelligence prediction API.